# 🚀 Chatterbox-TTS-Extended on Google Colab

Run the full Chatterbox-TTS-Extended UI on a free Colab GPU.

1. **Runtime → Change runtime type → T4 GPU**, then save.
2. Run the cells below in order.
3. Open the `https://….gradio.live` link printed by the **Launch** cell.

Colab sessions are temporary: mount Google Drive (step 3) or download your outputs (step 5) before the session ends.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi

In [ ]:
#@title 2. Clone the repo and install dependencies
#@markdown Takes a few minutes. Colab's preinstalled PyTorch is kept, since the CUDA build pinned in `requirements.txt` may not match Colab's driver.
#@markdown
#@markdown **Expected warnings:** pip may print `ERROR: pip's dependency resolver ... dependency conflicts` for Colab's preinstalled packages (e.g. `numba` for pytensor/cudf/cuml, `decorator` for moviepy, `jedi` for ipython). Chatterbox doesn't use them, so these are harmless; the install still succeeded.
%cd /content
![ -d Chatterbox-TTS-Extended ] || git clone https://github.com/enoky/Chatterbox-TTS-Extended.git
%cd /content/Chatterbox-TTS-Extended
!git pull --ff-only

!apt-get -qq install -y ffmpeg > /dev/null
!grep -vE '^(torch|torchaudio|torchcodec|auto-editor)==|^--extra-index-url' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt
# auto-editor is optional (only used when "Auto-Editor" is ticked in the UI) and may not install on every Python version.
!pip install -q auto-editor==29.3.1 || echo "auto-editor not installed; leave the Auto-Editor option off."

In [ ]:
#@title 3. (Optional) Save outputs to Google Drive
#@markdown Links the `output` folder to `MyDrive/Chatterbox-TTS-Extended/output` so files survive the session.
save_to_drive = True #@param {type:"boolean"}

import os
if save_to_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_out = '/content/drive/MyDrive/Chatterbox-TTS-Extended/output'
    os.makedirs(drive_out, exist_ok=True)
    local_out = '/content/Chatterbox-TTS-Extended/output'
    if os.path.islink(local_out):
        os.unlink(local_out)
    elif os.path.isdir(local_out):
        !cp -rn {local_out}/. {drive_out}/ && rm -rf {local_out}
    os.symlink(drive_out, local_out)
    print(f'Outputs will be saved to {drive_out}')

In [ ]:
#@title 4. Launch the UI
#@markdown Open the public `gradio.live` link printed below. The first generation downloads the models, which takes a few minutes. Stop this cell to shut the UI down.
%cd /content/Chatterbox-TTS-Extended
!python Chatter.py --share

In [ ]:
#@title 5. Download outputs as a zip
#@markdown Stop the Launch cell first, then run this one.
%cd /content/Chatterbox-TTS-Extended
!zip -qr /content/chatterbox_output.zip output/
from google.colab import files
files.download('/content/chatterbox_output.zip')